# Testing of random forest model transfer
This is a trial to test whether a model can be transferred from one classification purpose to other context.

In this step, the `MULTIPROBABILITY` output layers of the whole region (Sumatra in this case) have been generated and exported to local drive. This notebook's goal is to generate an LULC map from the pre-generated output layers in a smaller AOI

In [1]:
# !python -m pip install .. --quiet

import ee 
import luma_ge

# service_account_path = '../auth/ee-epstm2024.json'
# luma_ge.initialize_with_service_account(service_account_path)

ee.Authenticate()
ee.Initialize()

# Load AOI

In [2]:
import geemap

# Choose AOI

aoi = geemap.shp_to_ee('../data/AOI_Dempo.shp')

# Load default classification scheme

In [9]:
# Choose default classification scheme

from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "RESTORE+ Project"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

print(classification_df)

    ID             Land Cover Class Color Palette
0    1  Undisturbed dry-land forest       #006400
1    2  Logged-over dry-land forest       #228B22
2    3         Undisturbed mangrove       #4169E1
3    4         Logged-over mangrove       #87CEEB
4    5     Undisturbed swamp forest       #2E8B57
5    6     Logged-over swamp forest       #8FBC8F
6    7                 Agroforestry       #9ACD32
7    8            Plantation forest       #32CD32
8    9           Rubber monoculture       #8B4513
9   10         Oil palm monoculture       #FF8C00
10  11            Other monoculture       #DAA520
11  12                Grass/savanna       #ADFF2F
12  13                        Shrub       #90EE90
13  14                     Cropland       #FFFF00
14  15                   Settlement       #FF0000
15  16                 Cleared land       #D2B48C
16  17                    Waterbody       #0000FF


# Load training data and identify the classes that exist in the AOI from default scheme

In [10]:
# Load default training data from GEE asset and filter by AOI
# This is mainly to identify the existing classes inside the AOI and retrieve the class list information

from luma_ge.sample_data import SyncTrainData
import pandas as pd

TrainEePath = 'projects/ee-rg2icraf/assets/Indonesia_lulc_Sample'
TrainField = 'kelas'

TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=classification_df,
            aoi_geometry=aoi,
            training_ee_path=TrainEePath
        )

        # Set class field
TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)
        
        # Validate classes
TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, use_class_ids=True)
        
        # Check sufficiency
TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)
        
        # Filter by AOI
TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)

table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
            training_data=TrainDataDict.get('training_data'),
            landcover_df=TrainDataDict.get('landcover_df'),
            class_field=TrainDataDict.get('class_field')
        )

TrainDataFinal = TrainDataDict.get('training_data')

# Cross-check the class IDs in the filtered training data against the classification scheme
if TrainDataFinal is not None and hasattr(TrainDataFinal, 'columns') and 'kelas' in TrainDataFinal.columns:
    train_class_ids = pd.Series(TrainDataFinal['kelas'].dropna().astype(int).unique()).sort_values().tolist()
    scheme_df = classification_df[['ID', 'Land Cover Class']].copy()
    scheme_df['ID'] = scheme_df['ID'].astype(int)

    class_summary = []
    unmatched_ids = []

    for class_id in train_class_ids:
        scheme_match = scheme_df[scheme_df['ID'] == class_id]
        if not scheme_match.empty:
            class_name = scheme_match.iloc[0]['Land Cover Class']
            sample_count = int((TrainDataFinal['kelas'].astype(int) == class_id).sum())
            class_summary.append({
                'class_id': class_id,
                'class_name': class_name,
                'sample_count': sample_count
            })
        else:
            unmatched_ids.append(class_id)

    class_summary_df = pd.DataFrame(class_summary)
    display(class_summary_df)

    if unmatched_ids:
        print('Class IDs in training data but not present in classification_df:', unmatched_ids)
    else:
        print('All training class IDs are present in classification_df.')
else:
    print('No filtered training data available for class summary.')


,class_id,class_name,sample_count
0,1,Undisturbed dry-land forest,8
1,2,Logged-over dry-land forest,9
2,7,Agroforestry,24
3,9,Rubber monoculture,2
4,14,Cropland,1


All training class IDs are present in classification_df.


# Load the pre-generated multiprobability image stack
Ideally the image stack should have been merged as the result of the exported job from GEE is split into smaller areas

In [12]:
multiprobability_stack_path = '../data/temp/multiprobability_stack_example.tif'

multiprobability_stack = ee.Image(multiprobability_stack_path)

# Read the available band names from the loaded raster
band_names = None

if hasattr(multiprobability_stack, 'dims') and 'band' in multiprobability_stack.dims:
    band_names = [str(name) for name in multiprobability_stack['band'].values]
elif hasattr(multiprobability_stack, 'coords') and 'band' in multiprobability_stack.coords:
    band_names = [str(name) for name in multiprobability_stack.coords['band'].values]
elif hasattr(multiprobability_stack, 'band_names'):
    band_names = [str(name) for name in multiprobability_stack.band_names]
elif hasattr(multiprobability_stack, 'bands'):
    band_names = [str(band.name) for band in multiprobability_stack.bands]
else:
    band_names = [f'band_{i + 1}' for i in range(1)]

print('Available bands:')
for name in band_names:
    print(f' - {name}')

# Select only the bands whose names contain the class IDs from class_summary_df
if 'class_summary_df' in globals() and class_summary_df is not None and not class_summary_df.empty:
    class_ids = [str(int(cid)) for cid in class_summary_df['class_id'].astype(int).tolist()]
    selected_band_names = [name for name in band_names if any(class_id in str(name) for class_id in class_ids)]

    if not selected_band_names:
        selected_band_names = band_names[:min(len(band_names), len(class_ids))]

    print('Selected bands for AOI classes:')
    for name in selected_band_names:
        print(f' - {name}')

    if hasattr(multiprobability_stack, 'sel'):
        try:
            selected_stack = multiprobability_stack.sel(band=selected_band_names)
            print('Raster selection completed.')
            selected_stack
        except Exception as e:
            print(f'Band selection by name failed: {e}')
            print('Using the full stack instead.')
            multiprobability_stack
    else:
        print('Loaded raster object does not support band selection by name.')
else:
    print('class_summary_df is not available yet; run the training-data summary cell first.')


Available bands:
 - band_1
Selected bands for AOI classes:
 - band_1
Loaded raster object does not support band selection by name.
